In [1]:
#1. generate datasetx
#2  train the  classifier and save it
#3  detect the face and name it if it is already store in out dataset

# generate datase

In [2]:
import cv2

In [ ]:
def generate_dataset():
    face_classifier = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
    
    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) == 0:
            return None
        
        for (x, y, w, h) in faces:
            cropped_face = img[y:y+h, x:x+w]
        return cropped_face

    cap = cv2.VideoCapture(0)
    
    # Check if the camera is opened successfully
    if not cap.isOpened():
        print("Error: Camera not accessible")
        return
    
    # Ask user for the ID
    id = input("Enter the user ID: ")
    img_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame")
            break
        
        cropped_face = face_cropped(frame)
        if cropped_face is not None:
            img_id += 1
            face = cv2.resize(cropped_face, (200, 200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            file_name_path = "data/user." + str(id) + "." + str(img_id) + ".jpg"
            cv2.imwrite(file_name_path, face)
            cv2.putText(face, str(img_id), (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow("Cropped face", face)
        else:
            # Display the original frame if no face is detected
            cv2.imshow("Camera", frame)
        
        if cv2.waitKey(1) == 13 or int(img_id) == 100:  # Exit on Enter key or after collecting 100 images
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"Collecting samples for user ID {id} complete")

generate_dataset()


# Train the classifier and save it


In [4]:
import numpy as np
from PIL import Image
import os
import cv2

In [ ]:
def train_classifier(data_dir):
    # Get paths to all the images in the directory
    path = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
    faces = []
    ids = []

    for image in path:
        # Open the image and convert to grayscale
        img = Image.open(image).convert('L')
        imageNp = np.array(img, 'uint8')  # Convert image to NumPy array (uint8)

        # Extract the ID from the filename assuming the format is user.ID.xxx.jpg
        id = int(os.path.split(image)[1].split(".")[1])

        # Append face data and corresponding ID
        faces.append(imageNp)
        ids.append(id)

    # Convert ids to a NumPy array
    ids = np.array(ids)

    # Train the classifier and save it
    clf = cv2.face.LBPHFaceRecognizer_create()  # Ensure you have opencv-contrib-python installed
    clf.train(faces, ids)
    clf.write("classifier.xml")  # Save the trained model to a file
    print("Training complete and classifier saved as 'classifier.xml'")

# Call the function with the directory containing face images
train_classifier("data")


# Defect the face and name it if is already stored in our dataset

In [6]:
import cv2
import numpy as np
from PIL import Image
import os

In [ ]:
def draw_boundary(img,classifier,scaleFactor,minNeighbors,color,text,clf):
    gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    Features = classifier.detectMultiScale(gray_image, scaleFactor, minNeighbors)

    coords = []
    for(x,y,w,h) in Features:
        cv2.rectangle(img,(x,y),(x+w,y+h),color,2)
        id,pred= clf.predict(gray_image[y:y+h,x:x+w])
        condidence = int(100*(1-pred/300))

        if condidence>77:
            if id == 1:
                cv2.putText(img,"Aldi",(x,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.8,color,1,cv2.LINE_AA)
            if id == 2:
                cv2.putText(img,"Rizki",(x,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.8,color,1,cv2.LINE_AA)
            if id == 3:
                cv2.putText(img,"Ageng",(x,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.8,color,1,cv2.LINE_AA)
        else:
            cv2.putText(img,"UNKNOW",(x,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),1,cv2.LINE_AA)
        coords = [x,y,w,h]
    return coords
def recognize(img,clf,faceCascade):
    coords = draw_boundary(img,faceCascade,1.1,10,(255,255,255),"Face",clf)
    return  img
faceCascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
clf = cv2.face.LBPHFaceRecognizer_create()
clf.read("classifier.xml")

video_capture = cv2.VideoCapture(0)

while True:
    ret,img = video_capture.read()
    img=recognize(img,clf,faceCascade)
    cv2.imshow("face detection",img) 

    if cv2.waitKey(1) == 13:
        break

video_capture.release()
cv2.destroyAllWindows()

#cropface,convert to gray,resize,save to file--clasiffier
#train the classifier